# Exploración — Conectividad (SUBTEL)

Notebook de exploración para la limpieza de datos de internet fija por comuna, Región de Aysén.

Objetivo: validar la lógica de `load_conectividad_aysen()` antes de llevarla a `app/loaders.py`.

In [2]:
import pandas as pd
import plotly.express as px


## 1. ¿Por qué la hoja residencial y no el total?

La hoja `7.11.CO_FIJAS_COMUNA` (total de conexiones) incluye conexiones empresariales e institucionales. Esto genera valores muy altos en comunas pequeñas (ej. Lago Verde con más de 20.000 "conexiones" totales, cuando su población es de menos de 1.000 habitantes), lo que distorsiona la comparación con datos de población.

La hoja `7.11.1.CO_FIJAS_RES_COMUNA` (solo conexiones residenciales) da una relación mucho más coherente con el tamaño poblacional de cada comuna, y es más representativa de la pregunta que queremos responder: ¿cuántos hogares tienen acceso a internet?

## 2. Carga cruda del archivo

El header real está repartido en dos filas (año y mes), por eso se lee con `header=None` primero para inspeccionar la estructura.

In [ ]:
# Se define la ruta del dataset con las conexiones de internet fija por comuna desde el año 2015–2026
PATH = '../data/raw/subtel_internet_fija.xlsx'

# Se lee con pandas la hoja que contiene las conexiones desagregadas por comuna, sin encabezado
df_raw = pd.read_excel(PATH, sheet_name='7.11.1.CO_FIJAS_RES_COMUNA', header=None)

df_raw.shape

(400, 141)

In [4]:
# Revisamos el dataframe crudo
df_raw.head(20)

,0,1,2,3,4,5,6,7,8,9,...,131,132,133,134,135,136,137,138,139,140
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,SERVICIO ACCESO A INTERNET: TOTAL DE CONEXIONE...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,TOTAL DE CONEXIONES FIJAS RESIDENCIALES POR RE...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,<< VOLVER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,Año,2015,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
8,NaN,Región,Comuna,Ene,Feb,Mar,Abr,May,Jun,Jul,...,Jun,Jul,Ago,Sep,Oct,Nov,Dic,Ene,Feb,Mar
9,NaN,1,Alto Hospicio,5218,5188,5284,5405,5415,5467,5644,...,23435,23458,23608,23644,24204,24344,24452,24711,24298,23982


In [5]:
# Inspeccionar filas de encabezado: año en fila 7, mes en fila 8
df_raw.iloc[6:10, :10]

,0,1,2,3,4,5,6,7,8,9
6,NaN,<< VOLVER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,Año,2015,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Región,Comuna,Ene,Feb,Mar,Abr,May,Jun,Jul
9,NaN,1,Alto Hospicio,5218,5188,5284,5405,5415,5467,5644


In [6]:
# DEMOSTRACIÓN DEL PROBLEMA
# Lectura ingenua asumiendo una sola tabla continua: la columna 'comuna' siempre se toma de la posición de la Tabla 1 (col 2), incluso para 2019 en adelante

# Construir períodos desde fila 7 (año) y fila 8 (mes) para toda la hoja
row_anio = pd.to_numeric(df_raw.iloc[7, 3:], errors='coerce').ffill()
row_mes = df_raw.iloc[8, 3:]
periodos = [
    f'{int(a)}-{m}' if pd.notna(a) else str(m)
    for a, m in zip(row_anio, row_mes)
]

# Extraer bloque de Aysén usando siempre la columna 2 como 'comuna'
bloque = df_raw.iloc[280:291, 2:].copy()
bloque.columns = ['comuna'] + periodos[:bloque.shape[1]-1]
bloque = bloque[
    bloque['comuna'].notna() & (bloque['comuna'] != 'Sin clasificación')
]

# Aislar fila de Lago Verde para inspeccionar la transición problemática
lago_verde = bloque[bloque['comuna'] == 'Lago Verde'].iloc[0]

# El salto de 17 a ~13.000 entre dic-2018 y ene-2019 evidencia la lectura incorrecta
cols_transicion = ['2018-Oct', '2018-Nov', '2018-Dic', '2019-Ene', '2019-Feb', '2019-Mar']
print('Lago Verde — lectura INCORRECTA (tomandolo como solo una tabla):')
for col in cols_transicion:
    print(f'  {col}: {lago_verde.get(col, "N/D")}')

Lago Verde — lectura INCORRECTA (tomandolo como solo una tabla):
  2018-Oct: 17
  2018-Nov: 18
  2018-Dic: 17
  2019-Ene: 13330
  2019-Feb: 13551
  2019-Mar: 13509


## 3. Problema detectado: la hoja contiene DOS tablas, no una serie continua

Al revisar la serie de tiempo de comunas pequeñas (ej. Lago Verde), se observó un salto abrupto e inexplicable entre diciembre de 2018 (17 conexiones) y enero de 2019 (más de 13.000 conexiones). Ese salto no es creíble como crecimiento real.

Revisando el archivo directamente en Excel, se confirmó que **a partir de 2019 los datos están en otro bloque de columnas, con su propia columna de región y de comuna**, desplazado respecto al bloque 2015-2018. Si se ignora esto y se asume una sola tabla continua, las columnas de 2019 en adelante quedan leyendo los nombres de comuna del bloque viejo, mezclando datos de Aysén con datos de otras regiones (ej. Región Metropolitana, Los Lagos).

**Estructura real de la hoja:**

| Tabla | Período | Columna región | Columna comuna | Datos desde columna |
|---|---|---|---|---|
| 1 | 2015 - 2018 | 1 | 2 | 3 |
| 2 | 2019 - 2026 | 52 | 53 | 54 |

Y el bloque de la Región de Aysén está en **filas distintas** dentro de cada tabla (280-290 en la tabla 1, 259-269 en la tabla 2).

In [7]:
# Evidencia del problema: comparar el valor de Lago Verde leyendo siempre con la columna 'comuna' de la tabla 1 (columna 2), incluso para columnas de 2019
print('Diciembre 2018 (columna AY, idx 50):', df_raw.iloc[286, 50])
print('Enero 2019 leído con columna de comuna vieja (col 2) -> dato erróneo:')
print(df_raw.iloc[265:275, [1, 2, 54]])  # esto NO es Aysén, es Los Lagos

Diciembre 2018 (columna AY, idx 50): 17
Enero 2019 leído con columna de comuna vieja (col 2) -> dato erróneo:
      1             2      54
265  NaN  Puerto Montt     17
266  NaN  Puerto Octay    NaN
267  NaN  Puerto Varas    109
268  NaN     Puqueldón    NaN
269  NaN     Purranque    NaN
270  NaN       Puyehue  16425
271  NaN       Queilén    NaN
272  NaN       Quellón    NaN
273  NaN       Quemchi     28
274  NaN      Quinchao   3124


In [8]:
# Confirmación: dónde está realmente el bloque de Aysén en la tabla 2 (2019+)
idx_lago_verde_tabla2 = df_raw[df_raw.iloc[:, 53] == 'Lago Verde'].index.tolist()
print('Lago Verde en tabla 2 (columna 53), fila:', idx_lago_verde_tabla2)
df_raw.iloc[259:270, [52, 53, 54]]

Lago Verde en tabla 2 (columna 53), fila: [265]


,52,53,54
259,11,Aisén,3867
260,NaN,Chile Chico,291
261,NaN,Cisnes,865
262,NaN,Cochrane,312
263,NaN,Coyhaique,10847
264,NaN,Guaitecas,117
265,NaN,Lago Verde,17
266,NaN,O'Higgins,NaN
267,NaN,Río Ibáñez,109
268,NaN,Tortel,NaN


## 4. Extracción correcta: dos bloques, luego `concat`

Se extrae cada tabla por separado, con su propio rango de filas y columnas, y se combinan con `pd.concat` (apilando en el tiempo).

In [9]:
def extraer_bloque(df, fila_inicio, fila_fin, col_comuna, col_inicio_datos, periodos):
    # Índices numéricos de columnas a seleccionar: columna de comunas + columnas de datos
    cols = [col_comuna] + list(range(col_inicio_datos, col_inicio_datos + len(periodos)))
    
    # Extraer solo las filas de Aysén y las columnas relevantes
    bloque = df.iloc[fila_inicio:fila_fin, cols].copy()
    
    # Renombrar columnas con nombres legibles
    bloque.columns = ['comuna'] + periodos
    
    # Filtrar filas vacías y la fila 'Sin clasificación', dejando solo las 10 comunas
    bloque = bloque[bloque['comuna'].notna() & (bloque['comuna'] != 'Sin clasificación')]
    
    return bloque

# --- Tabla 1: 2015-2018 ---
# El año está en la fila 7 (columnas 3 a 50) y el mes en la fila 8
# ffill() rellena los NaN con el año anterior (el año solo aparece una vez por grupo de meses)
anio_1 = pd.to_numeric(df_raw.iloc[7, 3:51], errors='coerce').ffill()
mes_1 = df_raw.iloc[8, 3:51]
periodos_1 = [f'{int(a)}-{m}' for a, m in zip(anio_1, mes_1)]  # ej: ['2015-Ene', '2015-Feb', ...]
bloque_1 = extraer_bloque(df_raw, 280, 291, 2, 3, periodos_1)

# --- Tabla 2: 2019-2026 ---
# Segunda tabla dentro de la misma hoja, con su propio bloque de región/comuna
# El bloque de Aysén está en filas distintas (259-269) por el desfase de columnas detectado
anio_2 = pd.to_numeric(df_raw.iloc[7, 54:141], errors='coerce').ffill()
mes_2 = df_raw.iloc[8, 54:141]
periodos_2 = [f'{int(a)}-{m}' for a, m in zip(anio_2, mes_2)]
bloque_2 = extraer_bloque(df_raw, 259, 270, 53, 54, periodos_2)

# Verificar que ambas tablas contienen las mismas 10 comunas
print('Tabla 1 - comunas:', bloque_1['comuna'].tolist())
print('Tabla 2 - comunas:', bloque_2['comuna'].tolist())

Tabla 1 - comunas: ['Aisén', 'Chile Chico', 'Cisnes', 'Cochrane', 'Coyhaique', 'Guaitecas', 'Lago Verde', "O'Higgins", 'Río Ibáñez', 'Tortel']
Tabla 2 - comunas: ['Aisén', 'Chile Chico', 'Cisnes', 'Cochrane', 'Coyhaique', 'Guaitecas', 'Lago Verde', "O'Higgins", 'Río Ibáñez', 'Tortel']


In [18]:
# Convertir cada bloque de formato ancho a largo: una fila por comuna-período
largo_1 = bloque_1.melt(id_vars='comuna', var_name='periodo', value_name='conexiones')
largo_2 = bloque_2.melt(id_vars='comuna', var_name='periodo', value_name='conexiones')

# Apilar ambos períodos (2015-2018 y 2019-2026) en un solo DataFrame
df_largo = pd.concat([largo_1, largo_2], ignore_index=True)

# Separar '2015-Ene' en columnas 'anio' y 'mes'
df_largo[['anio', 'mes']] = df_largo['periodo'].str.split('-', expand=True)
df_largo['anio'] = df_largo['anio'].astype(int)

# Convertir conexiones a numérico (NaN donde no hay dato)
df_largo['conexiones'] = pd.to_numeric(df_largo['conexiones'], errors='coerce')

# Eliminar columna auxiliar 'periodo' y ordenar columnas
df_largo = df_largo.drop(columns='periodo')[['comuna', 'anio', 'mes', 'conexiones']]

# Normalizar el nombre de la comuna de Aysén de 'Aisén' a 'Aysén'
df_largo['comuna'] = df_largo['comuna'].replace({'Aisén' : 'Aysén'})
df_largo.head(10)

,comuna,anio,mes,conexiones
0,Aysén,2015,Ene,2549.0
1,Chile Chico,2015,Ene,276.0
2,Cisnes,2015,Ene,352.0
3,Cochrane,2015,Ene,220.0
4,Coyhaique,2015,Ene,7450.0
5,Guaitecas,2015,Ene,NaN
6,Lago Verde,2015,Ene,NaN
7,O'Higgins,2015,Ene,NaN
8,Río Ibáñez,2015,Ene,13.0
9,Tortel,2015,Ene,NaN


In [ ]:
# Verificación: la transición 2018 -> 2019 ya no debería tener saltos
df_largo.query("comuna == 'Lago Verde' and anio in [2018, 2019]")

,comuna,anio,mes,conexiones
366,Lago Verde,2018,Ene,18.0
376,Lago Verde,2018,Feb,18.0
386,Lago Verde,2018,Mar,17.0
396,Lago Verde,2018,Abr,17.0
406,Lago Verde,2018,May,17.0
416,Lago Verde,2018,Jun,17.0
426,Lago Verde,2018,Jul,17.0
436,Lago Verde,2018,Ago,17.0
446,Lago Verde,2018,Sep,17.0
456,Lago Verde,2018,Oct,17.0


In [ ]:
# Validaciones generales
print('Shape:', df_largo.shape)
print('Comunas:', df_largo['comuna'].nunique(), '->', sorted(df_largo['comuna'].unique()))
print('Rango de años:', df_largo['anio'].min(), '-', df_largo['anio'].max())
print('Nulos:', df_largo['conexiones'].isna().sum())

Shape: (1350, 4)
Comunas: 10 -> ['Aisén', 'Chile Chico', 'Cisnes', 'Cochrane', 'Coyhaique', 'Guaitecas', 'Lago Verde', "O'Higgins", 'Río Ibáñez', 'Tortel']
Rango de años: 2015 - 2026
Nulos: 198


In [ ]:
# Distribución de nulos por comuna
# Permite identificar qué comunas tienen datos incompletos y desde cuándo
nulos_por_comuna = df_largo.groupby('comuna')['conexiones'].apply(
    lambda x: x.isna().sum()
).rename('meses_sin_dato')

primer_dato = df_largo.groupby('comuna').apply(
    lambda x: x.dropna(subset=['conexiones'])['anio'].min()
).rename('primer_año_con_dato')

pd.concat([nulos_por_comuna, primer_dato], axis=1)

,meses_sin_dato,primer_año_con_dato
comuna,,
Aisén,0,2015
Chile Chico,0,2015
Cisnes,0,2015
Cochrane,0,2015
Coyhaique,0,2015
Guaitecas,8,2015
Lago Verde,8,2015
O'Higgins,91,2022
Río Ibáñez,0,2015


### Valores nulos

Los 198 valores nulos corresponden a comunas que no reportaban conexiones 
residenciales en los primeros años de la serie. Destacan O'Higgins y Tortel, 
que recién aparecen en 2022, lo que refleja su aislamiento geográfico extremo 
(Tortel no tiene acceso terrestre y O'Higgins es la comuna más austral de la región).

Estos nulos se mantienen en el dataset y son visibles en los gráficos como 
gaps en la serie temporal, lo que es informativo por sí mismo.

## 5. Exploración visual

In [ ]:
# Total regional por año (diciembre de cada año, como corte representativo)
anual = df_largo[df_largo['mes'] == 'Dic'].groupby('anio', as_index=False)['conexiones'].sum()

fig = px.line(anual, x='anio', y='conexiones',
              title='Conexiones de internet fija RESIDENCIAL — Región de Aysén (total regional, diciembre de cada año)')
fig.show()

In [ ]:
# Evolución por comuna (diciembre de cada año)
por_comuna_dic = df_largo[df_largo['mes'] == 'Dic']

fig = px.line(por_comuna_dic, x='anio', y='conexiones', color='comuna',
              title='Conexiones residenciales por comuna — Región de Aysén')
fig.show()

In [ ]:
# Comparación entre comunas en el último período disponible
ultimo_anio = df_largo['anio'].max()
ultimo_mes_disponible = df_largo[df_largo['anio'] == ultimo_anio]['mes'].iloc[-1]

ultimo_periodo = df_largo.query('anio == @ultimo_anio and mes == @ultimo_mes_disponible')
ultimo_periodo = ultimo_periodo.sort_values('conexiones', ascending=False)

fig = px.bar(ultimo_periodo, x='comuna', y='conexiones',
             title=f'Conexiones residenciales por comuna — {ultimo_mes_disponible} {ultimo_anio}')
fig.show()

In [ ]:
# Filtrar solo diciembres para la serie anual
dic = df_largo[df_largo['mes'] == 'Dic'].copy()

# Probar el gráfico con todas las comunas primero
fig = px.line(
    dic,
    x='anio',
    y='conexiones',
    color='comuna',
    markers=True,
    title='Conexiones de internet fija residencial por comuna — Región de Aysén',
    labels={
        'anio': 'Año',
        'conexiones': 'Conexiones residenciales',
        'comuna': 'Comuna'
    }
)
fig.show()

In [ ]:
# Probar el filtro por comuna (simula lo que hará el callback con Coyhaique por defecto)
comunas_seleccionadas = ['Coyhaique']

fig = px.line(
    dic.query('comuna in @comunas_seleccionadas'),
    x='anio',
    y='conexiones',
    color='comuna',
    markers=True,
    title='Conexiones de internet fija residencial por comuna — Región de Aysén',
    labels={
        'anio': 'Año',
        'conexiones': 'Conexiones residenciales',
        'comuna': 'Comuna'
    }
)
fig.show()